# M5b · Backfill `fact_harsh_event`

**Goal:** detect and persist harsh-driving events (brake / accel / corner) from
`staging.archive` accelerometer telemetry into `warehouse.fact_harsh_event`.

**Why this fact exists:** Project 1 (Driver Behavior Scoring & Risk
Classification) requires accelerometer-derived event counts. Without this fact,
the model has no signal for the most predictive risk driver.

**SQL file:** `sql/15_fact_harsh_event_incremental.sql`.
Bound parameters: `:window_start`, `:window_end`, `:etl_run_id`,
`:thresh_brake`, `:thresh_accel`, `:thresh_corner`, `:thresh_high`, `:thresh_extreme`.

**Tunable thresholds** (defaults from `config/pipeline.yaml`
`archive_thresholds`):

| Tier      | Default | Approx g (±2g sensor) |
|-----------|---------|------------------------|
| trigger   | 40      | 0.31 g                 |
| high      | 60      | 0.47 g                 |
| extreme   | 80      | 0.63 g                 |

**Exit criteria:**
- `etl_watermark.fact_harsh_event` advances to ≈ `MAX(staging.archive.date)`.
- Distribution of `event_type` and `severity` is reasonable
  (severe events should be rare; ~1% of all observations).

> **Run order:** before `M6c` (telemetry mart). Independent of fact_trip.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — config, thresholds, backfill plan

In [ ]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
THRESH     = cfg.get('archive_thresholds', {})
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)
print('thresholds =', THRESH)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date) FROM staging.archive")).scalar_one()
print('staging.archive max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('15_fact_harsh_event_incremental.sql')[:800], '...')


## 3. Execute — backfill in chunks

The archive table is large (~55M rows); we process in **smaller** chunks
than the trip facts because each ping does work (UNION ALL × 3 axes).


In [ ]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run

facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

# Override chunk size for the heavy archive table — keep transactions short.
HARSH_CHUNK_DAYS = 7

run_id = begin_run(mode='notebook:06_load_fact_harsh_event')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=HARSH_CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_harsh_event', run_id=run_id, window_end=next_cursor)
        progress.append({'window_start': cursor, 'window_end': next_cursor,
                         'rows_loaded': res.rows_loaded})
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise

print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


## 4. Inspect — watermark, type/severity distribution, sample events

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_harsh_event'"), conn)
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_harsh_event')).scalar_one()
    by_type = pd.read_sql(text('''
        SELECT event_type, severity, COUNT(*) AS n
          FROM warehouse.fact_harsh_event
         GROUP BY event_type, severity
         ORDER BY event_type, severity
    '''), conn)
    sample = pd.read_sql(text('''
        SELECT tenant_id, device_id, event_time, event_type, severity,
               x_axis, y_axis, speed_kmh, rpm
          FROM warehouse.fact_harsh_event
         ORDER BY event_time DESC LIMIT 10
    '''), conn)

print(f'fact_harsh_event total rows: {total:,}')
display(wm)
display(by_type)
display(sample)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — fact_harsh_event backfill complete.')
